In [4]:
# ===== Cell 1: Imports =====

import requests
import pandas as pd
import time
import os
from datetime import datetime
import torch
from torch.utils.data import Dataset
import numpy as np
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Subset
import torch.optim as optim


# ===== End of Cell 1 =====

In [6]:
# ===== Cell 2: Data acquisition =====

# ===== Constants for CNN-LSTM-60 =====
SEQ_LEN = 60    # M = 60 days lookback
INPUT_DIM = 7   # OHLCV (Open, High, Low, Close, Volume), dist from 14-day MA in %, 14-day ATR in %
CLASSES = 3     # Buy (2), Neutral (1), Sell (0)

class MasterStockDataset(Dataset):
    def __init__(self, master_file_path):
        """
        Loads the pre-processed master file from the sibling directory structure.
        Path: '../data/cnn_lstm_data/processed_data/master_dataset_M60.pt'
        """
        if not os.path.exists(master_file_path):
            raise FileNotFoundError(
                f"Master file not found at {master_file_path}. "
                "Check that you manually moved the .pt file to the 'processed_data' sibling folder."
            )

        print(f"📂 Loading CNN-LSTM-60 dataset from {master_file_path}...")

        # Load the structured list of dictionaries
        # weights_only=False is required for custom list/dict structures
        data_list = torch.load(master_file_path, weights_only=False)

        # Convert to bulk tensors for GPU efficiency
        # self.X shape: [N, 60, 7]
        # self.y shape: [N]
        self.X = torch.tensor(np.array([s['x'] for s in data_list]), dtype=torch.float32)
        self.y = torch.tensor(np.array([s['y'] for s in data_list]), dtype=torch.long)

        # Verification of the 7-feature input vector
        if self.X.shape[1] != SEQ_LEN or self.X.shape[2] != INPUT_DIM:
            print(f"⚠️ Warning: Data shape {self.X.shape[1:]} doesn't match expected ({SEQ_LEN}, {INPUT_DIM})")

        print(f"✅ Successfully loaded {len(self.y)} samples.")
        print(f"📊 Tensor Memory: {self.X.element_size() * self.X.nelement() / 1e6:.2f} MB")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ===== End of Cell 2 =====

In [7]:
# ===== Cell 3: Training Setup & Split =====

import sys
import os
from torch.utils.data import DataLoader, Subset
import torch.optim as optim

# 1. Path Management: Add 'src' to the path so we can import our model
sys.path.append(os.path.abspath(os.path.join('..')))
from src.models.cnn_lstm import StockCNNLSTM

# 2. Initialize Dataset
MASTER_PATH = "../data/cnn_lstm_data/processed_data/master_dataset_M60.pt"
full_dataset = MasterStockDataset(MASTER_PATH)

# 3. Perform Chronological Split (80/20)
train_size = int(0.8 * len(full_dataset))
indices = list(range(len(full_dataset)))

train_indices = indices[:train_size]
test_indices = indices[train_size:]

train_set = Subset(full_dataset, train_indices)
test_set = Subset(full_dataset, test_indices)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

# 4. Model, Loss, and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize with 7 features
model = StockCNNLSTM(input_dim=7, hidden_dim=64, output_dim=3).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"✅ Setup complete. Device: {device}")
print(f"📉 Training on {len(train_set)} samples | Testing on {len(test_set)} samples")

# ===== End of Cell 3 =====

📂 Loading CNN-LSTM-60 dataset from ../data/cnn_lstm_data/processed_data/master_dataset_M60.pt...
✅ Successfully loaded 71355 samples.
📊 Tensor Memory: 119.88 MB
✅ Setup complete. Device: cpu
📉 Training on 57084 samples | Testing on 14271 samples


In [ ]:
# ===== Cell 4: Training, Evaluation & Auto-Saving =====

import torch.nn as nn
from tqdm.auto import tqdm
import pandas as pd

# 1. Path Configuration
SAVE_DIR = "../scripts/best_runs/cnn_lstm"
os.makedirs(SAVE_DIR, exist_ok=True)
HISTORY_PATH = os.path.join(SAVE_DIR, "training_history.csv")
MODEL_PATH = os.path.join(SAVE_DIR, "best_cnn_lstm_weights.pth")

def train_model(model, train_loader, test_loader, criterion, optimizer, epochs=25):
    print(f"🚀 Training initiated on {device}...")
    best_test_acc = 0.0
    history = []

    for epoch in range(epochs):
        # --- Training Phase ---
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        for inputs, labels in train_bar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        avg_train_loss = running_loss / len(train_loader)
        train_acc = correct_train / total_train

        # --- Validation/Test Phase ---
        model.eval()
        test_loss = 0.0
        correct_test = 0
        total_test = 0
        class_correct = [0, 0, 0]
        class_total = [0, 0, 0]

        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total_test += labels.size(0)
                correct_test += (predicted == labels).sum().item()

                for i in range(len(labels)):
                    lbl = labels[i].item()
                    class_correct[lbl] += (predicted[i] == lbl).item()
                    class_total[lbl] += 1

        avg_test_loss = test_loss / len(test_loader)
        test_acc = correct_test / total_test

        # --- Save Best Model Weights ---
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            torch.save(model.state_dict(), MODEL_PATH)
            print(f"⭐ New Best Model Saved! (Test Acc: {test_acc:.2%})")

        # --- Log Progress ---
        epoch_stats = {
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "test_loss": avg_test_loss,
            "train_acc": train_acc,
            "test_acc": test_acc,
            "sell_acc": class_correct[0]/class_total[0] if class_total[0]>0 else 0,
            "neu_acc": class_correct[1]/class_total[1] if class_total[1]>0 else 0,
            "buy_acc": class_correct[2]/class_total[2] if class_total[2]>0 else 0
        }
        history.append(epoch_stats)

        # Save history to CSV immediately (can be plotted during run)
        pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

        print(f"\n[Epoch {epoch+1}] Loss|T: {avg_train_loss:.4f} V: {avg_test_loss:.4f}  Acc|T: {train_acc:.1%} V: {test_acc:.1%}")
        print(f"   Breakdown -> Sell: {epoch_stats['sell_acc']:.1%} | Neu: {epoch_stats['neu_acc']:.1%} | Buy: {epoch_stats['buy_acc']:.1%}")
        print("-" * 50)

# Run Training
train_model(model, train_loader, test_loader, criterion, optimizer, epochs=25)

# ===== End of Cell 4 =====